# Задача 3. Реакция первого порядка (химия)

Кинетика: $\dot c = -k c$ (реакция $A\to$ продукты).

Аналитика: $c(t) = c_0 e^{-kt}$.

**Задание:** реализуйте `loss_data`, `loss_physics`, `loss_ic`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

# Якорь PINN: ищем вверх от cwd каталог с workshop/lib/workshop_common.py
here = Path.cwd().resolve()
ROOT = None
for p in [here, *here.parents]:
    if (p / "workshop" / "lib" / "workshop_common.py").exists():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Не найден корень PINN (ожидался workshop/lib/workshop_common.py)")

sys.path.insert(0, str(ROOT / "workshop"))
from lib.workshop_common import DATA, MLP, derivative, load_xy_csv, plot_solution, set_seed

set_seed(42)


## Данные и аналитическое решение


In [ ]:
k = 0.8
c0 = 1.0

def analytical(t):
    t = np.asarray(t, dtype=np.float64)
    return c0 * np.exp(-k * t)

t_data, y_np = load_xy_csv(DATA / "first_order_kinetics.csv")
t_tensor = torch.tensor(t_data).view(-1, 1)
y_tensor = torch.tensor(y_np).view(-1, 1)
model = MLP(n_hidden=32)
print(f"точек данных: {len(t_data)}")


## Функции потерь (эталон)


In [ ]:
def loss_data(model, t, c):
    return torch.mean((model(t) - c) ** 2)

def loss_physics(model, t):
    t = t.clone().detach().requires_grad_(True)
    c = model(t)
    dc = derivative(c, t)
    return torch.mean((dc + k * c) ** 2)

def loss_ic(model):
    t0 = torch.zeros(1, 1, dtype=torch.float32)
    return (model(t0) - c0).pow(2).mean()


## Обучение


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
lambda_data, lambda_phys, lambda_ic = 1.0, 1.0, 1.0
num_epochs = 3000
print_every = 500

t_phys = torch.linspace(float(t_data.min()), float(t_data.max()), 100).view(-1, 1)

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    l_d = loss_data(model, t_tensor, y_tensor)
    l_p = loss_physics(model, t_phys)
    l_i = loss_ic(model)
    loss = lambda_data * l_d + lambda_phys * l_p + lambda_ic * l_i
    loss.backward()
    optimizer.step()
    if (epoch + 1) % print_every == 0:
        print(
            f"epoch {epoch+1}/{num_epochs}  loss={loss.item():.5f}  "
            f"data={l_d.item():.5f}  phys={l_p.item():.5f}  ic={l_i.item():.5f}"
        )


## Сравнение с аналитикой


In [ ]:
model.eval()
t_grid = np.linspace(float(t_data.min()), float(t_data.max()), 200)
with torch.no_grad():
    y_pred = model(torch.tensor(t_grid, dtype=torch.float32).view(-1, 1)).numpy().ravel()
plot_solution(t_data, y_np, t_grid, analytical(t_grid), y_pred, ylabel="c(t)", title="PINN: кинетика 1-го порядка")
